# SPOTnet — Step 07: Benchmark Evaluation

This notebook evaluates all models against the Arnold et al. baseline on the EndoRR test set.

**Models compared:**
- Arnold et al. (HSV baseline)
- SPOTnet Baseline (BCE + Dice)
- SPOTnet + Multi-Scale
- SPOTnet + Boundary Loss
- **SPOTnet (Full: MS + BL)**

**Metrics:** Accuracy, IoU, Precision, Recall, E-measure

## 1. Imports & Setup

In [ ]:
import os, glob, math
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BASE_CH = 32
TILE    = 384
OVERLAP = 96
print(f'Device: {DEVICE}')

# Test dataset — update paths as needed
IMG_DIR = 'dataset/EndoRR/test/image'
GT_DIR  = 'dataset/EndoRR/test/mask'

imgs = sorted(glob.glob(f'{IMG_DIR}/*.png') + glob.glob(f'{IMG_DIR}/*.jpg'))
gts  = sorted(glob.glob(f'{GT_DIR}/*.png'))
print(f'Test images: {len(imgs)}, GT masks: {len(gts)}')

## 2. Model & Inference

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, 1, 1, bias=False), nn.BatchNorm2d(cout), nn.SiLU(inplace=True),
            nn.Conv2d(cout, cout, 3, 1, 1, bias=False), nn.BatchNorm2d(cout), nn.SiLU(inplace=True))
    def forward(self, x): return self.net(x)

class DetUNet(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        self.e1 = ConvBlock(3, base)
        self.e2 = ConvBlock(base, base*2)
        self.e3 = ConvBlock(base*2, base*4)
        self.b  = ConvBlock(base*4, base*4)
        self.d2 = ConvBlock(base*4+base*2, base*2)
        self.d1 = ConvBlock(base*2+base, base)
        self.head = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        s1 = self.e1(x)
        s2 = self.e2(F.avg_pool2d(s1, 2))
        s3 = self.e3(F.avg_pool2d(s2, 2))
        b  = self.b(s3)
        x  = F.interpolate(b,  scale_factor=2, mode='bilinear', align_corners=False)
        x  = self.d2(torch.cat([x, s2], dim=1))
        x  = F.interpolate(x,  scale_factor=2, mode='bilinear', align_corners=False)
        x  = self.d1(torch.cat([x, s1], dim=1))
        return self.head(x)

def to_tensor(img):
    if not img.flags['C_CONTIGUOUS']: img = np.ascontiguousarray(img)
    return torch.from_numpy(img).float().permute(2,0,1) / 255.0

@torch.no_grad()
def tiled_inference(net, img_rgb, tile=384, overlap=96):
    net.eval()
    H, W = img_rgb.shape[:2]
    X = to_tensor(img_rgb).unsqueeze(0).to(DEVICE)
    stride = tile - overlap
    pad_h = (math.ceil((H-tile)/stride)*stride+tile-H) if H>tile else tile-H
    pad_w = (math.ceil((W-tile)/stride)*stride+tile-W) if W>tile else tile-W
    X = F.pad(X, (0,pad_w,0,pad_h), mode='reflect')
    Hp, Wp = X.shape[-2:]
    acc = torch.zeros((H,W), device=DEVICE)
    cnt = torch.zeros((H,W), device=DEVICE)
    for y in range(0, Hp-tile+1, stride):
        for x in range(0, Wp-tile+1, stride):
            pred = torch.sigmoid(net(X[:,:,y:y+tile,x:x+tile]))[0,0]
            ye, xe = min(y+tile,H), min(x+tile,W)
            acc[y:ye,x:xe] += pred[:ye-y,:xe-x]
            cnt[y:ye,x:xe] += 1
    return (acc/cnt.clamp(min=1)).cpu().numpy()

def load_model(ckpt_path):
    net = DetUNet(BASE_CH).to(DEVICE).eval()
    net.load_state_dict(torch.load(ckpt_path, map_location='cpu')['net'])
    return net

# Load all models
checkpoints = {
    'Baseline'     : 'runs/detector_baseline/best.pt',
    'Baseline+MS'  : 'runs/detector_multiscale_only/best.pt',
    'Baseline+BL'  : 'runs/detector_boundary_only/best.pt',
    'SPOTnet'      : 'runs/detector_spotnet/best.pt',
}

nets = {}
for name, path in checkpoints.items():
    if os.path.exists(path):
        nets[name] = load_model(path)
        print(f'✓ {name} loaded')
    else:
        print(f'✗ {name} not found: {path}')

## 3. Arnold et al. Baseline

In [ ]:
def arnold_detection(img_bgr, v_thr=200, s_thr=35):
    """Arnold et al. HSV-based specular detection."""
    hsv  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    mask = ((hsv[:,:,2] > v_thr) & (hsv[:,:,1] < s_thr)).astype(np.uint8) * 255
    k    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    return cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)

## 4. Metrics

In [ ]:
def e_measure(pred, gt):
    pred, gt = pred.astype(np.float32), gt.astype(np.float32)
    fm, gm   = pred - pred.mean(), gt - gt.mean()
    align    = 2 * fm * gm / (fm**2 + gm**2 + 1e-8)
    return (((align + 1)**2) / 4).mean()

def compute_metrics(pred, gt):
    pred_b = (pred > 127).astype(np.float32)
    gt_b   = (gt   > 127).astype(np.float32)
    tp = (pred_b * gt_b).sum()
    fp = (pred_b * (1-gt_b)).sum()
    fn = ((1-pred_b) * gt_b).sum()
    tn = ((1-pred_b) * (1-gt_b)).sum()
    return {
        'acc' : (tp+tn)/(tp+tn+fp+fn+1e-8),
        'iou' : tp/(tp+fp+fn+1e-8),
        'prec': tp/(tp+fp+1e-8),
        'rec' : tp/(tp+fn+1e-8),
        'em'  : e_measure(pred_b, gt_b)
    }

## 5. Run Benchmark

In [ ]:
results = {name: [] for name in list(nets.keys()) + ['Arnold']}

for img_path, gt_path in tqdm(zip(imgs, gts), total=len(imgs)):
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gt      = cv2.imread(gt_path, 0)

    # Arnold
    results['Arnold'].append(compute_metrics(arnold_detection(img_bgr), gt))

    # Neural models
    for name, net in nets.items():
        soft = tiled_inference(net, img_rgb, TILE, OVERLAP)
        pred = (soft > 0.5).astype(np.uint8) * 255
        results[name].append(compute_metrics(pred, gt))

print('\n' + '='*75)
print(f'{"Model":<16} {"Acc":<10} {"IoU":<10} {"Prec":<10} {"Rec":<10} {"E-m":<10}')
print('='*75)
for name, res in results.items():
    if res:
        print(f'{name:<16} '
              f'{np.mean([r["acc"]  for r in res]):.4f}±{np.std([r["acc"]  for r in res]):.3f}   '
              f'{np.mean([r["iou"]  for r in res]):.4f}±{np.std([r["iou"]  for r in res]):.3f}   '
              f'{np.mean([r["prec"] for r in res]):.4f}±{np.std([r["prec"] for r in res]):.3f}   '
              f'{np.mean([r["rec"]  for r in res]):.4f}±{np.std([r["rec"]  for r in res]):.3f}   '
              f'{np.mean([r["em"]   for r in res]):.4f}±{np.std([r["em"]   for r in res]):.3f}')

## 6. Visual Comparison

In [ ]:
idx = 0  # change to visualise different images
img_bgr = cv2.imread(imgs[idx])
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
gt      = cv2.imread(gts[idx], 0)

preds = {'Arnold': arnold_detection(img_bgr)}
for name, net in nets.items():
    soft = tiled_inference(net, img_rgb, TILE, OVERLAP)
    preds[name] = (soft > 0.5).astype(np.uint8) * 255

n_cols = len(preds) + 2
fig, axes = plt.subplots(1, n_cols, figsize=(4*n_cols, 4))

axes[0].imshow(img_rgb); axes[0].set_title('Original')
axes[1].imshow(gt, cmap='gray'); axes[1].set_title('GT')
for i, (name, pred) in enumerate(preds.items()):
    axes[i+2].imshow(pred, cmap='gray'); axes[i+2].set_title(name)
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()